In [1]:
import os
import sys
sys.path.append(os.path.join(os.getcwd(), ".."))
from utils import find_nearest_gene_atac_peaks, preprocessing_rna, preprocessing_atac, preprocessing_prot
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import muon as mu
from tqdm import tqdm
tqdm.pandas()
from pybiomart import Server

Dataset has been downloaded from:
https://www.covid19cellatlas.org/index.patient.html

In [2]:
adata = sc.read_h5ad("./data/haniffa21.processed.h5ad")
adata

AnnData object with n_obs × n_vars = 647366 × 24929
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'raw'

In [3]:
adata.X = np.array(adata.layers["raw"].todense())

In [4]:
random_state = np.random.RandomState()
indices = random_state.permutation(list(range(len(adata.obs))))
adata = adata[indices,:]

# RNA layer preprocessing

In [5]:
server = Server(host="http://www.ensembl.org")
dataset = server.marts["ENSEMBL_MART_ENSEMBL"].datasets["hsapiens_gene_ensembl"]
ensembl_gene_id_external_gene_name = dataset.query(attributes=["ensembl_gene_id", "external_gene_name"])
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name.rename(columns={"Gene stable ID": "gene_stable_id"})
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name[~ensembl_gene_id_external_gene_name["Gene name"].isnull()]
ensembl_gene_id_external_gene_name = ensembl_gene_id_external_gene_name.groupby('Gene name')['gene_stable_id'].agg(lambda x: '{' + ', '.join(f'"{item}"' for item in set(x)) + '}' if len(x) > 1 else x.iloc[0]).reset_index()

In [6]:
rna = adata[:, adata.var["feature_types"] == "Gene Expression"]

In [7]:
rna.var

,feature_types
MIR1302-2HG,Gene Expression
AL627309.1,Gene Expression
AL627309.3,Gene Expression
AL627309.2,Gene Expression
AL669831.2,Gene Expression
...,...
AC007325.2,Gene Expression
AL354822.1,Gene Expression
AC233755.2,Gene Expression
AC233755.1,Gene Expression


In [8]:
rna.var["gene_name"] = rna.var.index
rna.var["gene_name"] = rna.var["gene_name"].apply(lambda x: x.split(".")[0])
rna.var = rna.var.join(ensembl_gene_id_external_gene_name.set_index("Gene name"), on="gene_name")
rna.var

,feature_types,gene_name,gene_stable_id
MIR1302-2HG,Gene Expression,MIR1302-2HG,ENSG00000243485
AL627309.1,Gene Expression,AL627309,NaN
AL627309.3,Gene Expression,AL627309,NaN
AL627309.2,Gene Expression,AL627309,NaN
AL669831.2,Gene Expression,AL669831,NaN
...,...,...,...
AC007325.2,Gene Expression,AC007325,NaN
AL354822.1,Gene Expression,AL354822,NaN
AC233755.2,Gene Expression,AC233755,NaN
AC233755.1,Gene Expression,AC233755,NaN


In [9]:
rna = preprocessing_rna(rna)
rna

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user

AnnData object with n_obs × n_vars = 639599 × 24112
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types', 'gene_name', 'gene_stable_id', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap', 'log1p'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    layers: 'raw', 'counts'
    obsp: 'distances', 'connectivities'

# Prot layer preprocesssing

In [10]:
prot = adata[:, adata.var["feature_types"] == "Antibody Capture"]

In [11]:
prot.var["gene_stable_id"] = ['ENSG00000121594',
'ENSG00000114013',
'ENSG00000120217',
'ENSG00000197646',
'ENSG00000160223',
'ENSG00000169896',
'ENSG00000117586',
'ENSG00000125657',
'ENSG00000073008',
'ENSG00000130202',
'ENSG00000196776',
'ENSG00000125726',
'ENSG00000120949',
'ENSG00000101017',
'ENSG00000102245',
'ENSG00000169442',
'{"ENSG00000167286", "ENSG00000198851", "ENSG00000160654"}', 
'{"ENSG00000153563", "ENSG00000172116"}', 
'ENSG00000149294',
'ENSG00000177455',
'ENSG00000105383',
'ENSG00000140678',
'ENSG00000174059',
'ENSG00000048462',
'{"ENSG00000206503", "ENSG00000234745", "ENSG00000204525"}', 
'ENSG00000154096',
'ENSG00000157404',
'ENSG00000196549',
'ENSG00000081237',
'ENSG00000185291',
'ENSG00000173762',
'ENSG00000091409',
'ENSG00000183813',
'ENSG00000010610',
'ENSG00000026508',
'ENSG00000170458',
'ENSG00000203747',
'ENSG00000134460',
'ENSG00000081237',
'ENSG00000188389',
'ENSG00000181847',
'AB_Mouse IgG1_K_Iso', 
'AB_Mouse_IgG2a_K_Iso', 
'AB_Mouse_IgG2b_K_Iso', 
'AB_Rat_IgG2b_K_Iso', 
'ENSG00000156738',
'ENSG00000189430',
'ENSG00000183134',
'ENSG00000119888',
'ENSG00000261371',
'ENSG00000162493',
'ENSG00000076706',
'ENSG00000039068',
np.nan,
'ENSG00000110448',
'ENSG00000211689',
'ENSG00000186810',
'ENSG00000160791',
'ENSG00000143226',
'ENSG00000112486',
'ENSG00000160683',
'ENSG00000083457',
'ENSG00000110848',
'ENSG00000188404',
'ENSG00000126353',
'ENSG00000111796',
'ENSG00000163599',
'ENSG00000089692',
'ENSG00000139187',
'ENSG00000139193',
'ENSG00000185896',
'ENSG00000026103',
'ENSG00000204287',
'ENSG00000158481',
'ENSG00000150337',
'ENSG00000178726',
'ENSG00000158473',
'ENSG00000213809',
'ENSG00000124469',
'ENSG00000203710',
'ENSG00000109956',
'ENSG00000135077',
'ENSG00000186265',
'ENSG00000163600',
'ENSG00000116815',
'ENSG00000153283',
'ENSG00000138185',
'ENSG00000117560',
'ENSG00000168329',
'ENSG00000272398',
'ENSG00000117322',
'ENSG00000005844',
np.nan,
'ENSG00000007312',
'{"ENSG00000079385", "ENSG00000105388", "ENSG00000086548"}',
'ENSG00000122223',
'{"ENSG00000170180", "ENSG00000250361"}',
np.nan,
'ENSG00000088827',
'ENSG00000197992',
'ENSG00000173578',
'ENSG00000139626',
'ENSG00000159958',
'ENSG00000090339',
'ENSG00000174175',
np.nan,
'ENSG00000162692',
'ENSG00000100385',
'ENSG00000240505',
'ENSG00000179639',
'ENSG00000005961',
'ENSG00000049249',
'ENSG00000120659',
'ENSG00000177575',
'ENSG00000112149',
'ENSG00000186891',
'ENSG00000128052',
'ENSG00000077238',
'ENSG00000121966',
'ENSG00000116824',
'ENSG00000150637',
'ENSG00000150093',
'ENSG00000198178',
'ENSG00000164171',
'ENSG00000110651',
'ENSG00000168003',
np.nan,
'ENSG00000211898',
'ENSG00000160255',
'ENSG00000178562',
'ENSG00000205755',
'ENSG00000004468',
'ENSG00000168685',
'ENSG00000081237',
'ENSG00000196371',
'ENSG00000012124',
'ENSG00000072274',
'ENSG00000134258',
'ENSG00000197635',
'ENSG00000183625',
'ENSG00000038945',
'ENSG00000179776',
'ENSG00000158477',
'ENSG00000099250',
'ENSG00000135218',
'ENSG00000163749',
'ENSG00000116031',
'ENSG00000115232',
'ENSG00000135318',
'ENSG00000282506',
'{"ENSG000002266602", "ENSG00000282568"}',
'{"ENSG00000211716", "ENSG00000282204"}',
'ENSG00000211750',
'ENSG00000167613',
'ENSG00000173391',
'ENSG00000273661',
'ENSG00000007062',
'ENSG00000090659',
'ENSG00000167633',
'ENSG00000274143',
'ENSG00000204475',
'ENSG00000096264',
'ENSG00000163518',
'ENSG00000143297',
'ENSG00000026751',
'ENSG00000115884',
'ENSG00000002586',
'ENSG00000172322',
np.nan,
'ENSG00000102524',
'ENSG00000134539',
'ENSG00000117090',
'ENSG00000211592',
'ENSG00000131981',
'ENSG00000104972',
'ENSG00000104921',
'{"ENSG00000211675", "ENSG00000211677", "ENSG00000211679", "ENSG00000254029", "ENSG00000254030", "ENSG00000222037", "ENSG00000211685"}',
'ENSG00000206503',
'ENSG00000137507',
'ENSG00000168995',
'ENSG00000276405',
'ENSG00000085117',
'ENSG00000134256',
'ENSG00000103522',
'ENSG00000197405',
'ENSG00000204642',
'ENSG00000022556',
'ENSG00000128567',
'ENSG00000100031',
'ENSG00000105976',
'ENSG00000125735',
'ENSG00000215788']

In [12]:
prot.var["gene_name"] = prot.var.index
exclusion_features = ['AB_Mouse IgG1_K_Iso', 'AB_Mouse_IgG2a_K_Iso', 'AB_Mouse_IgG2b_K_Iso', 'AB_Rat_IgG2b_K_Iso']

In [13]:
prot = prot[:,~prot.var["gene_name"].isin(exclusion_features)]
prot

View of AnnData object with n_obs × n_vars = 647366 × 188
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types', 'gene_stable_id', 'gene_name'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    layers: 'raw'

In [15]:
prot = preprocessing_prot(prot)
prot

AnnData object with n_obs × n_vars = 647366 × 188
    obs: 'sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'full_clustering', 'initial_clustering', 'Resample', 'Collection_Day', 'Sex', 'Age_interval', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'patient_id'
    var: 'feature_types', 'gene_stable_id', 'gene_name'
    uns: 'hvg', 'leiden', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    layers: 'raw', 'counts'
    obsp: 'distances', 'connectivities'

# Create MuData and save

In [16]:
mdata = mu.MuData({"rna": rna, 
                   "prot": prot})

In [17]:
mu.pp.intersect_obs(mdata)

In [18]:
mdata.write("./data/Stephenson2021.h5mu")

A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user